# Event log construction

Every conversation becomes a process instance with two objects, the learner and the LLM, so the log is
object-centric (OCEL). The activity of an event is the first-level label of the utterance, consecutive
utterances of the same activity by the same agent are merged, and artificial timestamps keep the order
of the dialogue.

* **Input:** `labeled_conversations/<coding scheme>.csv`
* **Output:** `logs/log_<coding scheme>.csv` and `logs/ocel_<coding scheme>.jsonocel`
* **Check:** 159 cases per scheme, around 4,900 utterances merged into around 3,200 events


## DEPENDENCIES

In [5]:
import pandas as pd
import numpy as np
import pm4py
from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

## VARIABLES FOR EACH RUN

In [6]:
# Dataset to work on: 'data' is the study data, 'data_example' the synthetic example.
DATA = Path('..') / os.environ.get('DATASET_FOLDER', 'data')

IN_FOLDER = 'labeled_conversations'
CODING_SCHEMES = ['lin_dialogue_acts',
                  'acosta_epistemic_stance',
                  'del_bosque_cma']

OUT_FOLDER = 'logs'

ACTIVITY_COLUMN = 'type'  # first-level label

## FUNCTIONS

In [7]:
def convert_to_event_log(dataframe, label_column):
  # Consolidate two identical activities into one
  # A conversation is an activity_id + user_id pair
  conversation = dataframe['activity_id'] + '|' + dataframe['user_id']

  dataframe['same_session'] = np.equal(conversation, conversation.shift(+1))
  dataframe['same_role'] = np.equal(dataframe['role'], dataframe['role'].shift(+1))
  dataframe['same_label'] = np.equal(dataframe[label_column], dataframe[label_column].shift(+1))

  dataframe.fillna({'same_session': False,
               'same_role': False,
               'same_label': False}, inplace=True)

  # Work out when to consolidate the activities into one and when not to
  dataframe['consolidar'] = np.logical_and(np.logical_and(dataframe['same_session'], dataframe['same_role']), dataframe['same_label'])

  # Create an index to group by
  dataframe['group'] = (~dataframe['consolidar']).cumsum()

  # Create the new dataframe with the consolidated activities

  dataframe['sentence_en'] = dataframe['sentence_en'].astype(str)

  # Keep only the relevant columns
  df_consolidate = dataframe.groupby(['group']).agg({
      'activity_id': 'first',
      'user_id': 'first',
      'group': 'first',
      'role': 'first',
      'message_num': 'first',
      'utterance_num': 'first',
      label_column: 'first',
      'label': 'first',
      'type': 'first',
      'sentence_en': ' '.join,  # Concatenate activities
  }).reset_index(drop=True)


  # Create a new index that restarts for each conversation
  df_consolidate['session_id'] = (df_consolidate['activity_id'] + '|' + df_consolidate['user_id']).factorize()[0] + 1
  df_consolidate['index'] = df_consolidate.groupby('session_id').cumcount()
  # Create a fake timestamp
  df_consolidate['timestamp'] = pd.to_datetime('2024-01-01') + pd.to_timedelta(df_consolidate['index'], unit='m')

  return df_consolidate


def convert_to_ocpm(df, activity_column):

  dataframe = df.copy()
  dataframe = convert_to_event_log(dataframe, activity_column)
  dataframe['ocel:type:model'] = np.where(dataframe['role'] == 'model',
                                             '[\'' + dataframe['role'] + "_" + dataframe['user_id'] + '_' + dataframe['session_id'].astype(str) + '\']',
                                             ""
                                             )
  dataframe['ocel:type:user'] = np.where(dataframe['role'] == 'user',
                                            '[\'' + dataframe['role'] + "_" + dataframe['user_id'] + '_' + dataframe['session_id'].astype(str) + '\']',
                                             ""
                                             )

  dataframe['ocel:type:conversation'] = '[\'' + dataframe['session_id'].astype(str) + '\']'

  dataframe['ocel:eid'] = dataframe['group']
  dataframe['ocel:activity'] = dataframe[activity_column]
  dataframe['ocel:timestamp'] = dataframe['timestamp']

  dataframe['case_id'] = dataframe['session_id'].astype(str)

  dataframe = dataframe[['case_id', 'activity_id', 'ocel:eid', 'ocel:type:model',
                          'ocel:type:user', 'ocel:type:conversation', 'ocel:activity',
                          'ocel:timestamp', 'sentence_en', 'role', 'user_id', 'label', 'type']]
  return dataframe

## CREATING FILES

In [10]:
# Relative paths
path_in = DATA / IN_FOLDER
path_out = DATA / OUT_FOLDER

# Create the folder if it does not exist
path_out.mkdir(parents=True, exist_ok=True)

# Read dataframe
for coding_scheme in CODING_SCHEMES:
    df = pd.read_csv(path_in / (coding_scheme + '.csv'))

    # Create a log using the first-level labels as activity type
    out_log = convert_to_ocpm(df, ACTIVITY_COLUMN)

    out_log.to_csv(path_out / ('log_' + coding_scheme + '.csv'), index=False)

    # Export the same log as an OCEL file
    ocel = pm4py.convert.convert_log_to_ocel(out_log,
                                             activity_column='ocel:activity',
                                             timestamp_column='ocel:timestamp')
    #pm4py.write_ocel2_json(ocel, str(path_out / ('ocel_' + coding_scheme + '.jsonocel')))

    print(coding_scheme, ':', len(df), 'utterances ->', len(out_log), 'events in',
          out_log['case_id'].nunique(), 'conversations,',
          out_log['ocel:activity'].nunique(), 'activities')

out_log.tail(20)

lin_dialogue_acts : 4877 utterances -> 3272 events in 159 conversations, 13 activities
acosta_epistemic_stance : 4928 utterances -> 3146 events in 159 conversations, 5 activities
del_bosque_cma : 4963 utterances -> 3167 events in 159 conversations, 11 activities


,case_id,activity_id,ocel:eid,ocel:type:model,ocel:type:user,ocel:type:conversation,ocel:activity,ocel:timestamp,sentence_en,role,user_id,label,type
3147,158,asst_Get3WE5ozTjkHcdGhnqGGqgL,3148,['model_s95_158'],,['158'],Tutor Question Response,2024-01-01 00:15:00,Have you tried this approach?,model,s95,Tutor Question Response,Tutor Question Response
3148,158,asst_Get3WE5ozTjkHcdGhnqGGqgL,3149,['model_s95_158'],,['158'],Tutor Elicit Response,2024-01-01 00:16:00,I can guide you through a few examples if that...,model,s95,Tutor Elicit Response,Tutor Elicit Response
3149,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3150,['model_s96_159'],,['159'],Tutor Floor Initiating or Floor Mantaining,2024-01-01 00:00:00,"Hello! 😸 I am SIMBA, and I will help you think...",model,s96,Tutor Floor Initiating or Floor Mantaining,Tutor Floor Initiating or Floor Mantaining
3150,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3151,['model_s96_159'],,['159'],Tutor Elicit Response,2024-01-01 00:01:00,"In your opinion, what are the 2 to 5 main idea...",model,s96,Tutor Elicit Response,Tutor Elicit Response
3151,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3152,['model_s96_159'],,['159'],Tutor Floor Initiating or Floor Mantaining,2024-01-01 00:02:00,"Go ahead, share your thoughts! 🌟",model,s96,Tutor Floor Initiating or Floor Mantaining,Tutor Floor Initiating or Floor Mantaining
3152,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3153,,['user_s96_159'],['159'],Student Compound Contribution,2024-01-01 00:03:00,"Fluids follow state change curves, we mainly w...",user,s96,Student Compound Contribution,Student Compound Contribution
3153,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3154,['model_s96_159'],,['159'],Acknowledge Backchannel,2024-01-01 00:04:00,"Great, you've already grasped several concepts...",model,s96,Acknowledge Backchannel,Acknowledge Backchannel
3154,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3155,['model_s96_159'],,['159'],Tutor Elicit Response,2024-01-01 00:05:00,"To deepen your reflection, what do you think a...",model,s96,Tutor Elicit Response,Tutor Elicit Response
3155,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3156,['model_s96_159'],,['159'],Tutor Floor Initiating or Floor Mantaining,2024-01-01 00:06:00,"Moreover, you mention the critical point and t...",model,s96,Tutor Floor Initiating or Floor Mantaining,Tutor Floor Initiating or Floor Mantaining
3156,159,asst_Get3WE5ozTjkHcdGhnqGGqgL,3157,['model_s96_159'],,['159'],Tutor Elicit Response,2024-01-01 00:07:00,Can you explain why this state is particularly...,model,s96,Tutor Elicit Response,Tutor Elicit Response
